# Aura AI — Colab GPU Runtime
이 노트북은 Colab에서 Aura API와 Cloudflare Tunnel을 실행합니다. 런타임이 새로 시작되면 **런타임 → 모두 실행**을 누르면 환경을 다시 구성하고 서버를 재연결합니다.

Colab이 런타임 자체를 종료한 동안에는 어떤 코드도 실행될 수 없습니다. 새 런타임이 시작된 뒤 다시 실행해야 합니다.

In [ ]:
import os, sys, subprocess, pathlib, time, threading, signal, shutil
REPO = '/content/Aura-AI'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Workshop-Koharu/Aura-AI.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'python-multipart', 'pillow', 'psutil', 'requests'], check=True)
import torch, requests
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/AuraAI/data')
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
REPO_DATA = pathlib.Path(REPO) / 'data'
for name in ['text', 'image_text', 'audio_text', 'checkpoints', 'tokenizer']:
    src, dst = REPO_DATA / name, DRIVE_DATA / name
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink(): src.unlink()
    elif src.exists():
        for item in src.iterdir():
            target = dst / item.name
            if not target.exists(): shutil.copytree(item, target) if item.is_dir() else shutil.copy2(item, target)
        shutil.rmtree(src)
    src.symlink_to(dst, target_is_directory=True)
print('Drive data directory:', DRIVE_DATA)
os.environ['AURA_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['AURA_GPU_MEMORY_FRACTION'] = '0.33'
os.environ['AURA_CPU_THREADS'] = '2'
secret = userdata.get('AURA_GATEWAY_SECRET')
if not secret: raise RuntimeError('Colab Secrets에 AURA_GATEWAY_SECRET을 등록하고 Notebook access를 켜세요.')
os.environ['AURA_GATEWAY_SECRET'] = secret
print('Gateway secret loaded (hidden). Device:', os.environ['AURA_DEVICE'])


## 학습 데이터
권한이 있는 UTF-8 TXT 파일만 업로드하세요. 파일은 Google Drive에 저장됩니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
text_dir = DRIVE_DATA / 'text'
for name, content in uploaded.items():
    if name.lower().endswith('.txt'):
        (text_dir / pathlib.Path(name).name).write_bytes(content)
print('TXT count:', len(list(text_dir.glob('*.txt'))))


In [ ]:
from aura.training.runner import RUNNER
has_data = any((DRIVE_DATA / 'text').glob('*.txt')) or (DRIVE_DATA / 'image_text' / 'pairs.jsonl').exists() or (DRIVE_DATA / 'audio_text' / 'pairs.jsonl').exists()
if has_data: print('Training:', RUNNER.start())
else: print('학습 데이터가 없어 학습을 시작하지 않았습니다.')


In [ ]:
import subprocess, threading, requests, pathlib, re, urllib.request, os, time
API_LOG = open('/content/aura-api.log', 'a', buffering=1)
TUNNEL_LOG = open('/content/aura-tunnel.log', 'a', buffering=1)
CLOUDFLARED = '/usr/local/bin/cloudflared'
if not pathlib.Path(CLOUDFLARED).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
process_lock = threading.Lock()
processes = {'api': None, 'tunnel': None}
def start_api():
    p = processes['api']
    if p is None or p.poll() is not None:
        processes['api'] = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api.main:app', '--host', '127.0.0.1', '--port', '8000'], cwd=REPO, env=os.environ.copy(), stdout=API_LOG, stderr=subprocess.STDOUT)
        print('Aura API process restarted.')
def wait_api():
    for _ in range(60):
        try:
            r = requests.get('http://127.0.0.1:8000/api/health', timeout=2)
            if r.ok: return True
        except requests.RequestException: pass
        time.sleep(2)
    return False
def start_tunnel():
    p = processes['tunnel']
    if p is None or p.poll() is not None:
        processes['tunnel'] = subprocess.Popen([CLOUDFLARED, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT)
        print('Cloudflare Tunnel process started.')
def get_tunnel_url(timeout=180):
    log_path = pathlib.Path('/content/aura-tunnel.log')
    for _ in range(timeout // 2):
        try: log = log_path.read_text(errors='ignore')
        except OSError: log = ''
        matches = re.findall(r'https://[a-z0-9-]+\\.trycloudflare\\.com', log)
        if matches: return matches[-1]
        time.sleep(2)
    return None
def watchdog():
    while True:
        with process_lock:
            start_api()
            if wait_api(): start_tunnel()
        time.sleep(15)
with process_lock:
    start_api()
if not wait_api(): raise RuntimeError('Aura API가 시작되지 않았습니다. /content/aura-api.log 확인')
with process_lock: start_tunnel()
threading.Thread(target=watchdog, daemon=True).start()
PUBLIC_URL = get_tunnel_url()
print('AURA_INFERENCE_URL:', PUBLIC_URL or 'URL 생성 실패 — /content/aura-tunnel.log 확인')
print('Vercel 프로젝트 환경변수 AURA_INFERENCE_URL에 위 URL을 등록하세요. URL은 런타임 재시작 시 바뀔 수 있습니다.')
print('Health:', requests.get('http://127.0.0.1:8000/api/health', timeout=5).text)

threading.Thread(target=watchdog, daemon=True).start()


## 재접속 동작
- API 또는 Cloudflare Tunnel 프로세스가 중단되면 watchdog이 실행 중인 런타임 안에서 다시 시작합니다.
- Colab 런타임 자체가 종료되면 watchdog도 종료됩니다. 새 런타임에서 **런타임 → 모두 실행**을 실행하면 Drive 데이터와 체크포인트를 다시 연결하고 API/Tunnel을 재시작합니다.
- Quick Tunnel 주소는 새로 발급될 수 있으므로 Vercel의 `AURA_INFERENCE_URL`도 새 주소로 갱신해야 합니다.
- 상태 확인: `requests.get('http://127.0.0.1:8000/api/health').text`
